## Exercise 1: Set Up the Catalog Structure

You are a data engineer at **Lakeside University**, a mid-sized institution digitizing its academic operations. Your team is building a data platform in Azure Databricks and Unity Catalog to manage student records, course catalogues, and enrollment data.

In this exercise, you create the top-level namespace structure: one `catalog` and three `schemas` following the **medallion architecture** naming pattern.

Here's the full Unity Catalog structure you will create:

```txt

Unity Catalog
└── edu_dev  [catalog]
    │   tags: environment=development | university=lakeside | data_classification=internal
    │
    ├── bronze  [schema]
    │   └── raw_files/  [managed volume]
    │       └── students.csv
    │
    ├── silver  [schema]
    │   ├── students  [table]
    │   │   ├── student_id       BIGINT  NOT NULL  ← PK
    │   │   ├── first_name       STRING
    │   │   ├── last_name        STRING
    │   │   ├── email            STRING
    │   │   ├── enrollment_year  INT
    │   │   ├── program          STRING
    │   │   └── phone_number     STRING  (added in Exercise 6)
    │   │
    │   ├── courses  [table]
    │   │   ├── course_id    BIGINT  NOT NULL  ← PK
    │   │   ├── course_name  STRING
    │   │   ├── department   STRING
    │   │   └── credits      INT
    │   │
    │   ├── enrollments  [table]
    │   │   ├── enrollment_id  BIGINT  NOT NULL  ← PK
    │   │   ├── student_id     BIGINT            → FK → students.student_id
    │   │   ├── course_id      BIGINT            → FK → courses.course_id
    │   │   ├── semester       STRING
    │   │   └── grade          DECIMAL(4,2)
    │   │
    │   ├── vw_student_enrollments  [view]
    │   │   └── joins students + courses + enrollments
    │   │
    │   └── get_grade_classification(grade)  [function]
    │       └── returns STRING  (A / B / C / D / F)
    │
    └── gold  [schema]
        └── vw_department_enrollment_stats  [materialized view]
            └── aggregates enrollments + courses
                (department, total_enrollments, avg_grade, distinct_students)
```

%md
## **Before starting the lab, you need a Unity Catalog volume to store the file**. Follow these steps to create one:

- In the Databricks workspace sidebar, click Catalog.
- In the Catalog Explorer,right side click on + icon and create a Catalog ( DP750_Lab_03 ), type =standard & default storage.
- Then Create expand Catalog and generate a new schema (MySchema) by clicking on + icon and create Volume.
- Click the ⋮ menu next to default, then select Create > Volume. (Bronze,Silver,Gold) repeat same step to create all 3 schema
- Enter lab_data as the volume name, leave the type as Managed, and click Create.

### Task 1.2 — Create medallion schemas

Within the `edu_dev` catalog, create three schemas that implement the **medallion architecture**:

| Schema | Purpose |
|--------|---------|
| `bronze` | Raw ingested data — unmodified source files |
| `silver` | Cleaned, validated, and enriched data |
| `gold` | Aggregated, analytics-ready data |

### Code Explanation

```python
spark.sql("CREATE SCHEMA IF NOT EXISTS DP750_Lab_03.bronze")
```

- `spark` → SparkSession object used to interact with Spark.
- `.sql()` → Executes an SQL statement from PySpark.
- `CREATE SCHEMA` → Creates a new schema (database namespace).
- `IF NOT EXISTS` → Prevents an error if the schema already exists.
- `DP750_Lab_03` → Catalog name.
- `bronze` → Schema name, typically used to store raw ingested data in the Medallion Architecture.

### Purpose
Creates the **bronze schema inside the `DP750_Lab_03` catalog** only if it does not already exist.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS DP750_Lab_03.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS DP750_Lab_03.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS DP750_Lab_03.gold")

DataFrame[]

## Exercise 2: Create Tables with Constraints

With the catalog structure in place, create the core tables for Lakeside University's academic data in the `silver` schema. You'll define **primary key** and **foreign key** constraints to document the data model. Note that in Unity Catalog, these constraints are informational — they help optimize queries but do not enforce referential integrity at write time.

### Task 2.1 — Create the `students` table

Create a managed Delta table `edu_dev.silver.students` to store student records with the following columns:

| Column | Type | Description |
|--------|------|-------------|
| `student_id` | BIGINT | Unique student identifier (primary key) |
| `first_name` | STRING | Student's first name |
| `last_name` | STRING | Student's last name |
| `email` | STRING | University email address |
| `enrollment_year` | INT | Year the student enrolled |
| `program` | STRING | Degree program |


### Code Explanation

- `spark.sql()` → Executes SQL statements from PySpark.
- `CREATE TABLE IF NOT EXISTS` → Creates the table only if it doesn't already exist.
- `dp750_lab_03.silver.students` → Name of the table in the Silver layer.
- `student_id BIGINT NOT NULL` → Mandatory column storing large integer values.
- `first_name`, `last_name`, `email`, `program` → String columns to store student details.
- `enrollment_year INT` → Integer column storing the enrollment year.
- `PRIMARY KEY (student_id)` → Declares `student_id` as the primary key.
- `USING DELTA` → Stores the table in Delta Lake format.

**Purpose:** Creates a Delta table named `students` in the Silver layer with a primary key on `student_id`.

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dp750_lab_03.silver.students (
  student_id BIGINT NOT NULL,
  first_name STRING,
  last_name STRING,
  email STRING,
  enrollment_year INT,
  program STRING,
  CONSTRAINT students_pk PRIMARY KEY (student_id)
) USING DELTA
""")

DataFrame[]

### Task 2.2 — Create the `courses` table

Create `edu_dev.silver.courses` with the following columns:

| Column | Type | Description |
|--------|------|-------------|
| `course_id` | BIGINT | Unique course identifier (primary key) |
| `course_name` | STRING | Full name of the course |
| `department` | STRING | Academic department offering the course |
| `credits` | INT | Number of academic credits |



In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dp750_lab_03.silver.courses (
  course_id BIGINT NOT NULL,
  course_name STRING,
  department STRING,
  credits INT,
  CONSTRAINT courses_pk PRIMARY KEY (course_id)
) USING DELTA
""")

DataFrame[]

### Task 2.3 — Create the `enrollments` table with foreign keys

Create `edu_dev.silver.enrollments` to record which students are enrolled in which courses. This table must reference both `students` and `courses` via foreign key constraints.

| Column | Type | Description |
|--------|------|-------------|
| `enrollment_id` | BIGINT | Unique enrollment record (primary key) |
| `student_id` | BIGINT | References `students.student_id` |
| `course_id` | BIGINT | References `courses.course_id` |
| `semester` | STRING | Academic semester (e.g., `Spring 2023`) |
| `grade` | DECIMAL(4,2) | Numerical grade on a 0.0–10.0 scale |



In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dp750_lab_03.silver.enrollments (
  enrollment_id BIGINT NOT NULL,
  student_id BIGINT,
  course_id BIGINT,
  semester STRING,
  grade DECIMAL(4,2),
  CONSTRAINT enrollments_pk PRIMARY KEY (enrollment_id),
  CONSTRAINT enrollments_student_fk FOREIGN KEY (student_id) REFERENCES dp750_lab_03.silver.students(student_id),
  CONSTRAINT enrollments_course_fk FOREIGN KEY (course_id) REFERENCES dp750_lab_03.silver.courses(course_id)
) USING DELTA
""")

DataFrame[]

### Sample data — run this cell

The following cell inserts sample data into your three tables. **Run it without modification** before continuing to Exercise 3.

In [0]:
%sql
-- Populate students
INSERT INTO dp750_lab_03.silver.students VALUES
(1001, 'Emma',   'Watson',    'emma.watson@lakeside.edu',    2022, 'Computer Science'),
(1002, 'Liam',   'Ahmed',     'liam.ahmed@lakeside.edu',     2023, 'Data Science'),
(1003, 'Sofia',  'Chen',      'sofia.chen@lakeside.edu',     2021, 'Mathematics'),
(1004, 'James',  'Okonkwo',   'james.okonkwo@lakeside.edu',  2022, 'Computer Science'),
(1005, 'Aisha',  'Patel',     'aisha.patel@lakeside.edu',    2023, 'Data Science'),
(1006, 'Noah',   'Yamamoto',  'noah.yamamoto@lakeside.edu',  2020, 'Mathematics'),
(1007, 'Olivia', 'Kowalski',  'olivia.kowalski@lakeside.edu',2021, 'Computer Science'),
(1008, 'Carlos', 'Reyes',     'carlos.reyes@lakeside.edu',   2022, 'Data Science'),
(1009, 'Mei',    'Lindqvist', 'mei.lindqvist@lakeside.edu',  2023, 'Mathematics'),
(1010, 'Jake',   'OBrien',    'jake.obrien@lakeside.edu',    2020, 'Computer Science');

-- Populate courses
INSERT INTO dp750_lab_03.silver.courses VALUES
(101, 'Introduction to Programming',   'Computer Science', 3),
(102, 'Data Structures',               'Computer Science', 4),
(103, 'Statistics for Data Science',   'Data Science',     3),
(104, 'Machine Learning Fundamentals', 'Data Science',     4),
(105, 'Calculus I',                    'Mathematics',      4),
(106, 'Linear Algebra',                'Mathematics',      3);

-- Populate enrollments
INSERT INTO dp750_lab_03.silver.enrollments VALUES
(1,  1001, 101, 'Spring 2023', 8.50),
(2,  1001, 102, 'Fall 2023',   7.80),
(3,  1002, 103, 'Spring 2023', 9.20),
(4,  1002, 104, 'Fall 2023',   6.50),
(5,  1003, 105, 'Spring 2022', 7.00),
(6,  1003, 106, 'Fall 2022',   8.00),
(7,  1004, 101, 'Fall 2022',   6.80),
(8,  1005, 103, 'Spring 2023', 9.00),
(9,  1006, 105, 'Fall 2020',   7.50),
(10, 1007, 102, 'Spring 2022', 8.20),
(11, 1008, 104, 'Fall 2023',   7.30),
(12, 1009, 106, 'Spring 2023', 8.80),
(13, 1010, 101, 'Fall 2021',   5.50),
(14, 1004, 102, 'Spring 2023', 7.10),
(15, 1005, 104, 'Fall 2023',   8.90);

num_affected_rows,num_inserted_rows
15,15


## Exercise 3: Create Views

Views provide a virtual layer over your tables, simplifying access to joined data and pre-computing expensive aggregations. In this exercise, you create a **standard view** for combined enrollment records and a **materialized view** in the `gold` schema to serve the Lakeside University analytics team with pre-aggregated department statistics.

### Task 3.1 — Create a standard view

Create a view `edu_dev.silver.vw_student_enrollments` that presents a combined, human-readable view of student enrollment records. It should include:

- Student full name (concatenate `first_name` and `last_name` with a space between them)
- Student `email`
- `course_name`
- `department`
- `semester`
- `grade`


In [0]:
spark.sql("""
CREATE VIEW IF NOT EXISTS dp750_lab_03.silver.vw_student_enrollments AS
SELECT
  CONCAT(s.first_name, ' ', s.last_name) AS full_name,
  s.email,
  c.course_name,
  c.department,
  e.semester,
  e.grade
FROM dp750_lab_03.silver.enrollments e
JOIN dp750_lab_03.silver.students s ON e.student_id = s.student_id
JOIN dp750_lab_03.silver.courses c ON e.course_id = c.course_id
""")

DataFrame[]

### Task 3.2 — Create a materialized view for department statistics

Create a materialized view `edu_dev.gold.vw_department_enrollment_stats` that pre-computes the following per academic department:

| Output column | Description |
|--------------|-------------|
| `department` | Academic department name |
| `total_enrollments` | Total number of enrollment records |
| `avg_grade` | Average grade, rounded to 2 decimal places |
| `distinct_students` | Number of unique students enrolled |


# View vs Materialized View

| Feature | View | Materialized View |
|---------|------|-------------------|
| Storage | Does not store data | Stores data physically |
| Data Retrieval | Executes the underlying query every time | Reads precomputed data |
| Performance | Slower for complex queries | Faster for reporting and analytics |
| Data Freshness | Always shows the latest data | May contain stale data until refreshed |
| Refresh Required | No | Yes (`REFRESH MATERIALIZED VIEW`) |
| Disk Space | Requires minimal space | Consumes additional storage |
| Best Use Case | Simple abstraction, security, and reusable queries | Dashboards, aggregations, and frequently accessed reports |

### Example
- **View:** A virtual table showing all active customers. Every query fetches the latest data from the base tables.
- **Materialized View:** A precomputed monthly sales summary stored on disk and refreshed periodically for faster reporting.

This code will give error if you run on Serverless compute or classic compute . 

# Change the compute to serverless compute and run the below cell.

In [0]:
%sql
CREATE MATERIALIZED VIEW IF NOT EXISTS dp750_lab_03.gold.vw_department_enrollment_stats AS
SELECT
  c.department,
  COUNT(*) AS total_enrollments,
  ROUND(AVG(e.grade), 2) AS avg_grade,
  COUNT(DISTINCT e.student_id) AS distinct_students
FROM dp750_lab_03.silver.enrollments e
JOIN dp750_lab_03.silver.courses c ON e.course_id = c.course_id
GROUP BY c.department


result
The operation was successfully executed.


**✅ Test your materialized view** — Run the following query to verify it returns per-department aggregates:

You should see one row per academic department with columns for `total_enrollments`, `avg_grade`, and `distinct_students`.

In [0]:
%sql
SELECT * FROM dp750_lab_03.gold.vw_department_enrollment_stats;

department,total_enrollments,avg_grade,distinct_students
Mathematics,4,7.83,3
Data Science,5,8.18,3
Computer Science,6,7.32,4


## Exercise 4: Create a Volume and Read Files

Volumes extend Unity Catalog governance to files, not just structured tables. In this exercise, you create a **managed volume** in the `bronze` schema to serve as a landing zone for raw CSV files — simulating files arriving from Lakeside University's external student information system.

### Task 4.1 — Create a managed volume

Create a managed volume named `raw_files` in the `edu_dev.bronze` schema. This will serve as the landing area for raw data files.
- Go to bronze Schema Under catalog - dp750_lab_03  
- Click on + icon and create volumne named as ( raw_files ), Managed Storage and Select your Catalog name and schema as bronze from Drop down


### Task 4.2 — Write a CSV file to the volume

The following Python cell writes a sample student CSV file directly into your new volume. **Run it without modification.**

## Switch Back to Serverless pool since you will be running Python code below and can't be run on SQL warehouse

In [0]:
# Write a sample student CSV file to the volume — run this cell without modification
csv_content = """student_id,first_name,last_name,email,enrollment_year,program
1001,Emma,Watson,emma.watson@lakeside.edu,2022,Computer Science
1002,Liam,Ahmed,liam.ahmed@lakeside.edu,2023,Data Science
1003,Sofia,Chen,sofia.chen@lakeside.edu,2021,Mathematics
1004,James,Okonkwo,james.okonkwo@lakeside.edu,2022,Computer Science
1005,Aisha,Patel,aisha.patel@lakeside.edu,2023,Data Science
1006,Noah,Yamamoto,noah.yamamoto@lakeside.edu,2020,Mathematics
1007,Olivia,Kowalski,olivia.kowalski@lakeside.edu,2021,Computer Science
1008,Carlos,Reyes,carlos.reyes@lakeside.edu,2022,Data Science
1009,Mei,Lindqvist,mei.lindqvist@lakeside.edu,2023,Mathematics
1010,Jake,OBrien,jake.obrien@lakeside.edu,2020,Computer Science"""

dbutils.fs.put("/Volumes/dp750_lab_03/bronze/raw_files/students.csv", csv_content, overwrite=True)
print("students.csv written to /Volumes/dp750_lab_03/bronze/raw_files/")

Wrote 692 bytes.
students.csv written to /Volumes/dp750_lab_03/bronze/raw_files/


### Task 4.3 — Read the CSV from the volume into a DataFrame

Read the CSV file you just placed in the volume into a Spark DataFrame and display its contents. The file is at `/Volumes/edu_dev/bronze/raw_files/students.csv`.


### Code Explanation

- `spark.read` → Uses Spark's DataFrameReader to read data from a file.
- `.csv()` → Specifies that the source file is in CSV format.
- `"/Volumes/dp750_lab_03/bronze/raw_files/students.csv"` → Path to the CSV file.
- `header=True` → Treats the first row of the file as column names.
- `inferSchema=True` → Automatically detects the data types of the columns.
- `df =` → Stores the loaded data in a DataFrame named `df`.
- `display(df)` → Displays the contents of the DataFrame in Databricks.

**Purpose:** Reads the `students.csv` file into a Spark DataFrame and displays its contents.

In [0]:
df = spark.read.csv("/Volumes/dp750_lab_03/bronze/raw_files/students.csv", header=True, inferSchema=True)
display(df)

student_id,first_name,last_name,email,enrollment_year,program
1001,Emma,Watson,emma.watson@lakeside.edu,2022,Computer Science
1002,Liam,Ahmed,liam.ahmed@lakeside.edu,2023,Data Science
1003,Sofia,Chen,sofia.chen@lakeside.edu,2021,Mathematics
1004,James,Okonkwo,james.okonkwo@lakeside.edu,2022,Computer Science
1005,Aisha,Patel,aisha.patel@lakeside.edu,2023,Data Science
1006,Noah,Yamamoto,noah.yamamoto@lakeside.edu,2020,Mathematics
1007,Olivia,Kowalski,olivia.kowalski@lakeside.edu,2021,Computer Science
1008,Carlos,Reyes,carlos.reyes@lakeside.edu,2022,Data Science
1009,Mei,Lindqvist,mei.lindqvist@lakeside.edu,2023,Mathematics
1010,Jake,OBrien,jake.obrien@lakeside.edu,2020,Computer Science


## Exercise 5: Create a Reusable SQL Function

Unity Catalog allows you to create **governed, reusable functions** that encapsulate business logic. Instead of repeating grade conversion logic in every query, you'll write it once as a SQL function and call it from any notebook or query in the workspace.

### Task 5.1 — Create a grade classification function

Create a SQL scalar function `edu_dev.silver.get_grade_classification` that accepts a `grade DECIMAL(4,2)` and returns a `STRING` letter classification based on the scale used at Lakeside University:

| Grade range | Classification |
|-------------|----------------|
| ≥ 8.5 | `'A'` |
| ≥ 7.0 | `'B'` |
| ≥ 5.5 | `'C'` |
| ≥ 4.0 | `'D'` |
| < 4.0 | `'F'` |



In [0]:
%sql
CREATE FUNCTION dp750_lab_03.silver.get_grade_classification(grade DECIMAL(4,2))
RETURNS STRING
RETURN CASE
  WHEN grade >= 8.5 THEN 'A'
  WHEN grade >= 7.0 THEN 'B'
  WHEN grade >= 5.5 THEN 'C'
  WHEN grade >= 4.0 THEN 'D'
  ELSE 'F'
END;

### Task 5.2 — Test the function

Query `edu_dev.silver.enrollments` and apply your new function to produce a `grade_classification` column for each row. Include: `enrollment_id`, `student_id`, `course_id`, `semester`, `grade`, and `grade_classification`.


In [0]:
df = spark.sql("""
SELECT
  enrollment_id,
  student_id,
  course_id,
  semester,
  grade,
  dp750_lab_03.silver.get_grade_classification(grade) AS grade_classification
FROM dp750_lab_03.silver.enrollments
""")
display(df)

enrollment_id,student_id,course_id,semester,grade,grade_classification
1,1001,101,Spring 2023,8.50,A
2,1001,102,Fall 2023,7.80,B
3,1002,103,Spring 2023,9.20,A
4,1002,104,Fall 2023,6.50,C
5,1003,105,Spring 2022,7.00,B
6,1003,106,Fall 2022,8.00,B
7,1004,101,Fall 2022,6.80,C
8,1005,103,Spring 2023,9.00,A
9,1006,105,Fall 2020,7.50,B
10,1007,102,Spring 2022,8.20,B


## Exercise 6: DDL Operations

Unity Catalog objects evolve over time as requirements change. In this exercise, you use `ALTER` statements to:
1. Extend the `students` table with a new column.
2. Apply governance **tags** to the `edu_dev` catalog.

### Task 6.1 — Add a column to the students table

The student registration system has been updated to capture phone numbers. Add a nullable `phone_number` column of type `STRING` to `edu_dev.silver.students`.


In [0]:
spark.sql("""
ALTER TABLE dp750_lab_03.silver.students
ADD COLUMN phone_number STRING
""")

DataFrame[]

### Task 6.2 — Apply tags to the catalog

Apply the following tags to the `edu_dev` catalog to support governance and discoverability:

| Tag key | Tag value |
|---------|----------|
| `environment` | `'development'` |
| `university` | `'lakeside'` |
| `data_classification` | `'internal'` |


In [0]:
spark.sql("""
ALTER CATALOG dp750_lab_03 SET TAGS ('environment' = 'development', 'university' = 'lakeside', 'data_classification' = 'internal')
""")

DataFrame[]

### Task 6.3 — Verify your work

Run the following cells to confirm your catalog structure, table schemas, and tags are all in place. Compare the output with what you created throughout this lab.

In [0]:
%sql
-- Show all tables and views in the silver schema
SHOW TABLES IN dp750_lab_03.silver;

database,tableName,isTemporary
silver,courses,false
silver,enrollments,false
silver,students,false
silver,vw_student_enrollments,false


In [0]:
%sql
-- Describe the students table — verify the new phone_number column is present
DESCRIBE TABLE dp750_lab_03.silver.students;

col_name,data_type,comment
student_id,bigint,null
first_name,string,null
last_name,string,null
email,string,null
enrollment_year,int,null
program,string,null
phone_number,string,null


In [0]:
%sql
-- Show catalog details
DESCRIBE CATALOG EXTENDED dp750_lab_03;

info_name,info_value
Catalog Name,dp750_lab_03
Comment,
Owner,admin@vbhvprksh1outlook.onmicrosoft.com
Catalog Type,Regular
Created By,admin@vbhvprksh1outlook.onmicrosoft.com
Created At,2026-06-25 AD at 02:35:11 UTC
Updated By,admin@vbhvprksh1outlook.onmicrosoft.com
Updated At,2026-06-25 AD at 02:59:00 UTC
Storage Root,
Storage Location,


In [0]:
%sql 
-- Query tags via information schema (structured way)
SELECT *
FROM system.information_schema.catalog_tags
WHERE catalog_name = 'dp750_lab_03';

catalog_name,tag_name,tag_value
dp750_lab_03,environment,development
dp750_lab_03,university,lakeside
dp750_lab_03,data_classification,internal


---

## Next steps

You have completed all notebook exercises. Return to the **lab instructions** and continue with:

## - **Exercise**: Configure an AI/BI Genie Space (optional)